In [1]:
import numpy as np
import rasterio
import pandas as pd
from rasterio.windows import Window, from_bounds
from dask.distributed import Client, LocalCluster

# --- Input paths ---
#base = "/mnt/d/Mikayla/Github/typology/data"

paths = {
    'secveg': r"/home/gisuser/code/data/sites_selection_data/input/MB_c10_secFveg_2023_P.tif",  #secveg_2024_m_P.tif",
    'glad_height': r"/home/gisuser/code/data/sites_selection_data/input/GLAD_Canopy_Height_SaoPaulo_250km_P.tif",
    'veg_age': r"/home/gisuser/code/data/sites_selection_data/input/MB_c9_secFveg_age_2023_P.tif",
    'typology': r"/home/gisuser/code/data/sites_selection_data/input/20-24_combined_rc.tif",
    'patch_area': r"/home/gisuser/code/data/sites_selection_data/input/2024_area_1km.tif",
    'patch_edge': r"/home/gisuser/code/data/sites_selection_data/input/2024_edge_1km.tif",
    'patch_num': r"/home/gisuser/code/data/sites_selection_data/input/2024_pn_1km.tif",
    'shdi': r"/home/gisuser/code/data/sites_selection_data/input/brazil_coverage_2024_P_C_C.tif"
}

# --- Output paths ---
out_class_path = r"/home/gisuser/code/data/sites_selection_data/output/forest_typology_class.tif"
out_table_path = r"/home/gisuser/code/data/sites_selection_data/output/forest_typology_table.csv"

In [2]:
# --- Initialize Dask cluster ---
cluster = LocalCluster(n_workers=2, threads_per_worker=6, memory_limit='20GB') #eds threats per work = 12, mks laptop = 6
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

Dashboard: http://127.0.0.1:8787/status


In [4]:
# --- Get processing extent from typology (smallest raster) ---
with rasterio.open(paths['typology']) as ref:
    ref_bounds = ref.bounds
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_height = ref.height
    ref_width = ref.width
    ref_profile = ref.profile.copy()
    ref_nodata = ref.nodata

print(f"Processing extent: {ref_bounds}")
print(f"Dimensions: {ref_height} rows x {ref_width} cols")
print(f"Typology NoData value: {ref_nodata}")

# --- Output profile based on typology extent ---
ref_profile.update(dtype='uint8', nodata=0, count=1)

# Create output raster
with rasterio.open(out_class_path, "w", **ref_profile) as dst:
    pass

# --- Define processing function for each chunk ---
def process_chunk(row_start, row_end, ref_width, ref_transform, ref_nodata, paths):
    import numpy as np
    import rasterio
    import pandas as pd
    from rasterio.windows import Window, from_bounds

    chunk_height = row_end - row_start
    out_window = Window(0, row_start, ref_width, chunk_height)
    chunk_transform = rasterio.windows.transform(out_window, ref_transform)
    chunk_bounds = rasterio.transform.array_bounds(chunk_height, ref_width, chunk_transform)

    # Load chunk from all rasters
    chunk = {}
    for name, path in paths.items():
        with rasterio.open(path) as src:
            win = from_bounds(*chunk_bounds, transform=src.transform)
            win = win.round_offsets().round_lengths()
            data = src.read(1, window=win)
            if data.shape != (chunk_height, ref_width):
                tmp = np.zeros((chunk_height, ref_width), dtype=data.dtype)
                min_h = min(data.shape[0], chunk_height)
                min_w = min(data.shape[1], ref_width)
                tmp[:min_h, :min_w] = data[:min_h, :min_w]
                data = tmp
            chunk[name] = data

    # Mask: only process where typology has valid data
    if ref_nodata is not None:
        valid_mask = chunk['typology'] != ref_nodata
    else:
        valid_mask = ~np.isnan(chunk['typology'].astype(float))

    # Skip chunk entirely if no valid pixels
    if not valid_mask.any():
        return None, None

    # Reclassify age: 255 -> 0
    chunk['veg_age'] = np.where(chunk['veg_age'] == 255, 0, chunk['veg_age'])

    # Classification (only where typology is valid)
    result = np.zeros((chunk_height, ref_width), dtype=np.uint8)

    # 1. Mature: primary veg (class 2) + height >= 21m + age >= 30
    mature = valid_mask & (chunk['secveg'] == 2) & (chunk['glad_height'] >= 21) & (chunk['veg_age'] >= 30)
    result[mature] = 1

    # 2. Natural Regen: secondary regrowth (class 5) + age >= 5
    natural_regen = valid_mask & (chunk['secveg'] == 5) & (chunk['veg_age'] >= 5) & (result == 0)
    result[natural_regen] = 4

    # 3. Non-restored: deforestation of primary (class 4) or secondary (class 6)
    non_restored = valid_mask & ((chunk['secveg'] == 4) | (chunk['secveg'] == 6)) & (result == 0)
    result[non_restored] = 2

    # Extract table rows for classified pixels
    rows_idx, cols_idx = np.where(result > 0)
    df = None
    if len(rows_idx) > 0:
        global_rows = rows_idx + row_start
        xs, ys = rasterio.transform.xy(ref_transform, global_rows, cols_idx)

        df = pd.DataFrame({
            'x': xs,
            'y': ys,
            'class': result[rows_idx, cols_idx],
            'age': chunk['veg_age'][rows_idx, cols_idx],
            'glad_height': chunk['glad_height'][rows_idx, cols_idx],
            'secveg_class': chunk['secveg'][rows_idx, cols_idx],
            'typology': chunk['typology'][rows_idx, cols_idx],
            'patch_area': chunk['patch_area'][rows_idx, cols_idx],
            'patch_edge': chunk['patch_edge'][rows_idx, cols_idx],
            'patch_number': chunk['patch_num'][rows_idx, cols_idx],
            'shdi': chunk['shdi'][rows_idx, cols_idx]
        })

    return result, df

# --- Submit chunks to Dask workers ---
chunk_size = 2048
futures = []

for row_start in range(0, ref_height, chunk_size):
    row_end = min(row_start + chunk_size, ref_height)
    future = client.submit(
        process_chunk,
        row_start, row_end, ref_width, ref_transform, ref_nodata, paths
    )
    futures.append((row_start, row_end, future))

print(f"Submitted {len(futures)} chunks to Dask cluster...")

# --- Collect results and write output ---
all_rows = []

for row_start, row_end, future in futures:
    result, df = future.result()

    if result is not None:
        # Write classified chunk to output raster
        out_window = Window(0, row_start, ref_width, row_end - row_start)
        with rasterio.open(out_class_path, "r+") as dst:
            dst.write(result, 1, window=out_window)

    if df is not None:
        all_rows.append(df)

    print(f"  Collected rows {row_start}-{row_end} / {ref_height}")

# --- Combine and save CSV ---
print("Combining table...")
df = pd.concat(all_rows, ignore_index=True)
df.to_csv(out_table_path, index=False)

print(f"Table saved: {out_table_path}")
print(f"Done!")
print(f"Classified raster: {out_class_path}")
print(f"Table exported: {out_table_path}")
print(f"Total classified pixels: {len(df)}")
print(f"  Mature (1): {(df['class'] == 1).sum()}")
print(f"  Non-restored (2): {(df['class'] == 2).sum()}")
print(f"  Natural Regen (4): {(df['class'] == 4).sum()}")

# --- Clean up ---
client.close()
cluster.close()

Processing extent: BoundingBox(left=956371.638759929, bottom=561893.2195062793, right=1882850.0036588889, top=1288790.5110649331)
Dimensions: 22796 rows x 29055 cols
Typology NoData value: 15.0
Submitted 12 chunks to Dask cluster...
  Collected rows 0-2048 / 22796
  Collected rows 2048-4096 / 22796
  Collected rows 4096-6144 / 22796
  Collected rows 6144-8192 / 22796
  Collected rows 8192-10240 / 22796
  Collected rows 10240-12288 / 22796
  Collected rows 12288-14336 / 22796
  Collected rows 14336-16384 / 22796
  Collected rows 16384-18432 / 22796
  Collected rows 18432-20480 / 22796
  Collected rows 20480-22528 / 22796
  Collected rows 22528-22796 / 22796
Combining table...
Table saved: /home/gisuser/code/data/sites_selection_data/output/forest_typology_table.csv
Done!
Classified raster: /home/gisuser/code/data/sites_selection_data/output/forest_typology_class.tif
Table exported: /home/gisuser/code/data/sites_selection_data/output/forest_typology_table.csv
Total classified pixels: 350

In [8]:
# Read the FULL secveg raster (small enough to test?)
with rasterio.open(paths['secveg']) as src:
    print("dtype:", src.dtypes[0])
    print("shape:", src.shape)
    print("transform:", src.transform)
    
    # Read directly by pixel window, not from_bounds
    sample = src.read(1, window=Window(0, 0, 5000, 5000))
    print("Direct window - unique:", np.unique(sample))

# Now compare with typology
with rasterio.open(paths['typology']) as ref:
    print("Typology transform:", ref.transform)
    print("Typology shape:", ref.shape)

dtype: uint16
shape: (54627, 56328)
transform: | 31.89, 0.00, 428800.32|
| 0.00,-31.89, 1832719.88|
| 0.00, 0.00, 1.00|
Direct window - unique: [  0 256]
Typology transform: | 31.89, 0.00, 956371.64|
| 0.00,-31.89, 1288790.51|
| 0.00, 0.00, 1.00|
Typology shape: (22796, 29055)


In [5]:
# Read secveg at the TYPOLOGY extent instead of top-left corner
from rasterio.windows import from_bounds

with rasterio.open(paths['typology']) as ref:
    bounds = ref.bounds
    print("Typology bounds:", bounds)

with rasterio.open(paths['secveg']) as src:
    win = from_bounds(*bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    print("Window in secveg:", win)
    sample = src.read(1, window=win)
    print("Unique values at typology extent:", np.unique(sample))
    print("Shape:", sample.shape)


Typology bounds: BoundingBox(left=956371.638759929, bottom=561893.2195062793, right=1882850.0036588889, top=1288790.5110649331)
Window in secveg: Window(col_off=-2586, row_off=-2521, width=29055, height=22796)
Unique values at typology extent: [  0   1   2   3   4   6   7 255]
Shape: (17705, 16211)


In [6]:
with rasterio.open(paths['secveg']) as src:
    win = from_bounds(*bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    sample = src.read(1, window=win)
    
    print("Value counts:")
    vals, counts = np.unique(sample, return_counts=True)
    for v, c in zip(vals, counts):
        print(f"  {v}: {c:,} pixels")


Value counts:
  0: 43,057,120 pixels
  1: 105,744,423 pixels
  2: 46,235,360 pixels
  3: 9,862,806 pixels
  4: 107,815 pixels
  6: 161,373 pixels
  7: 4,542,785 pixels
  255: 77,304,073 pixels


In [12]:
with rasterio.open(paths['secveg']) as src:
    win = from_bounds(*bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    secveg_sample = src.read(1, window=win)

with rasterio.open(paths['veg_age']) as src:
    win = from_bounds(*bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    age_sample = src.read(1, window=win)

# What ages do class 3 pixels have?
class3_ages = age_sample[secveg_sample == 3]
class3_ages_clean = class3_ages[class3_ages != 255]  # remove nodata
print("Class 3 pixels with valid age:", len(class3_ages_clean))
print("Age range:", class3_ages_clean.min(), "-", class3_ages_clean.max())
print("Age distribution:")
vals, counts = np.unique(class3_ages_clean, return_counts=True)
for v, c in zip(vals[:20], counts[:20]):
    print(f"  Age {v}: {c:,}")


Class 3 pixels with valid age: 6430987
Age range: 1 - 38
Age distribution:
  Age 1: 6,040
  Age 2: 1,694
  Age 3: 228,540
  Age 4: 151,157
  Age 5: 176,482
  Age 6: 174,151
  Age 7: 177,223
  Age 8: 171,717
  Age 9: 251,607
  Age 10: 219,259
  Age 11: 200,663
  Age 12: 205,717
  Age 13: 226,220
  Age 14: 231,054
  Age 15: 193,523
  Age 16: 209,782
  Age 17: 168,498
  Age 18: 188,867
  Age 19: 174,947
  Age 20: 186,726


In [8]:
from rasterio.windows import from_bounds

with rasterio.open(paths['typology']) as ref:
    ref_bounds = ref.bounds

with rasterio.open(paths['secveg']) as src:
    print("File:", paths['secveg'])
    print("dtype:", src.dtypes[0])
    print("nodata:", src.nodata)
    print("shape:", src.shape)
    print("bands:", src.count)
    
    # Read at typology extent
    win = from_bounds(*ref_bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    full_sample = src.read(1, window=win)
    
    print("ALL unique values:", np.unique(full_sample))
    print("Value counts:")
    vals, counts = np.unique(full_sample, return_counts=True)
    for v, c in zip(vals, counts):
        print(f"  {v}: {c:,} pixels ({c/full_sample.size*100:.2f}%)")

File: /home/gisuser/code/data/sites_selection_data/input/secveg_2024_m_P.tif
dtype: uint16
nodata: 256.0
shape: (54627, 56328)
bands: 1
ALL unique values: [  0   1   2   3   4   6   7 256]
Value counts:
  0: 202,725,653 pixels (30.61%)
  1: 229,390,558 pixels (34.63%)
  2: 80,645,384 pixels (12.18%)
  3: 18,211,841 pixels (2.75%)
  4: 160,390 pixels (0.02%)
  6: 155,303 pixels (0.02%)
  7: 9,687,942 pixels (1.46%)
  256: 121,360,709 pixels (18.32%)


In [3]:
#rolled back secveg class issue and kept edits to 256 mask
# follow up with this code

# --- Get processing extent from typology (smallest raster) ---
with rasterio.open(paths['typology']) as ref:
    ref_bounds = ref.bounds
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_height = ref.height
    ref_width = ref.width
    ref_profile = ref.profile.copy()
    ref_nodata = ref.nodata

print(f"Processing extent: {ref_bounds}")
print(f"Dimensions: {ref_height} rows x {ref_width} cols")
print(f"Typology NoData value: {ref_nodata}")

# --- Output profile based on typology extent ---
ref_profile.update(dtype='uint8', nodata=0, count=1)

# Create output raster
with rasterio.open(out_class_path, "w", **ref_profile) as dst:
    pass

# --- Define processing function for each chunk ---
def process_chunk(row_start, row_end, ref_width, ref_transform, ref_nodata, paths):
    import numpy as np
    import rasterio
    import pandas as pd
    from rasterio.windows import Window, from_bounds

    chunk_height = row_end - row_start
    out_window = Window(0, row_start, ref_width, chunk_height)
    chunk_transform = rasterio.windows.transform(out_window, ref_transform)
    chunk_bounds = rasterio.transform.array_bounds(chunk_height, ref_width, chunk_transform)

    # Load chunk from all rasters
    chunk = {}
    for name, path in paths.items():
        with rasterio.open(path) as src:
            win = from_bounds(*chunk_bounds, transform=src.transform)
            win = win.round_offsets().round_lengths()
            data = src.read(1, window=win)
            if data.shape != (chunk_height, ref_width):
                tmp = np.zeros((chunk_height, ref_width), dtype=data.dtype)
                min_h = min(data.shape[0], chunk_height)
                min_w = min(data.shape[1], ref_width)
                tmp[:min_h, :min_w] = data[:min_h, :min_w]
                data = tmp
            chunk[name] = data

    # Mask: only process where typology has valid data
    if ref_nodata is not None:
        valid_mask = chunk['typology'] != ref_nodata
    else:
        valid_mask = ~np.isnan(chunk['typology'].astype(float))

    # Skip chunk entirely if no valid pixels
    if not valid_mask.any():
        return None, None

    # Reclassify age: 255 -> 0 (NoData/ignore)
    chunk['veg_age'] = np.where(chunk['veg_age'] == 255, 0, chunk['veg_age'])

    # Mask: secveg 256 = NoData
    secveg_valid = chunk['secveg'] != 256

    # Classification (only where typology AND secveg are valid)
    result = np.zeros((chunk_height, ref_width), dtype=np.uint8)

    # 1. Mature: primary veg (class 2) + height >= 21m + age >= 30
    mature = valid_mask & secveg_valid & (chunk['secveg'] == 2) & (chunk['glad_height'] >= 21) & (chunk['veg_age'] >= 30)
    result[mature] = 1

    # 2. Natural Regen: secondary veg regrowth (class 3) + age >= 5
    natural_regen = valid_mask & secveg_valid & (chunk['secveg'] == 3) & (chunk['veg_age'] >= 5) & (result == 0)
    result[natural_regen] = 4

    # 3. Non-restored: deforestation of primary (class 4) or secondary (class 6)
    non_restored = valid_mask & secveg_valid & ((chunk['secveg'] == 4) | (chunk['secveg'] == 6)) & (result == 0)
    result[non_restored] = 2

    # Extract table rows for classified pixels
    rows_idx, cols_idx = np.where(result > 0)
    df = None
    if len(rows_idx) > 0:
        global_rows = rows_idx + row_start
        xs, ys = rasterio.transform.xy(ref_transform, global_rows, cols_idx)

        df = pd.DataFrame({
            'x': xs,
            'y': ys,
            'class': result[rows_idx, cols_idx],
            'age': chunk['veg_age'][rows_idx, cols_idx],
            'glad_height': chunk['glad_height'][rows_idx, cols_idx],
            'secveg_class': chunk['secveg'][rows_idx, cols_idx],
            'typology': chunk['typology'][rows_idx, cols_idx],
            'patch_area': chunk['patch_area'][rows_idx, cols_idx],
            'patch_edge': chunk['patch_edge'][rows_idx, cols_idx],
            'patch_number': chunk['patch_num'][rows_idx, cols_idx],
            'shdi': chunk['shdi'][rows_idx, cols_idx]
        })

    return result, df

# --- Submit chunks to Dask workers ---
chunk_size = 2048
futures = []

for row_start in range(0, ref_height, chunk_size):
    row_end = min(row_start + chunk_size, ref_height)
    future = client.submit(
        process_chunk,
        row_start, row_end, ref_width, ref_transform, ref_nodata, paths
    )
    futures.append((row_start, row_end, future))

print(f"Submitted {len(futures)} chunks to Dask cluster...")

# --- Collect results and write output ---
all_rows = []

for row_start, row_end, future in futures:
    result, df = future.result()

    if result is not None:
        out_window = Window(0, row_start, ref_width, row_end - row_start)
        with rasterio.open(out_class_path, "r+") as dst:
            dst.write(result, 1, window=out_window)

    if df is not None:
        all_rows.append(df)

    print(f"  Collected rows {row_start}-{row_end} / {ref_height}")

# --- Combine and save CSV ---
print("Combining table...")
df = pd.concat(all_rows, ignore_index=True)
df.to_csv(out_table_path, index=False)

print(f"Done!")
print(f"Classified raster: {out_class_path}")
print(f"Table exported: {out_table_path}")
print(f"Total classified pixels: {len(df)}")
print(f"  Mature (1): {(df['class'] == 1).sum()}")
print(f"  Non-restored (2): {(df['class'] == 2).sum()}")
print(f"  Natural Regen (4): {(df['class'] == 4).sum()}")

# --- Clean up ---
client.close()
cluster.close()

Processing extent: BoundingBox(left=956371.638759929, bottom=561893.2195062793, right=1882850.0036588889, top=1288790.5110649331)
Dimensions: 22796 rows x 29055 cols
Typology NoData value: 15.0
Submitted 12 chunks to Dask cluster...
  Collected rows 0-2048 / 22796
  Collected rows 2048-4096 / 22796
  Collected rows 4096-6144 / 22796
  Collected rows 6144-8192 / 22796
  Collected rows 8192-10240 / 22796
  Collected rows 10240-12288 / 22796
  Collected rows 12288-14336 / 22796
  Collected rows 14336-16384 / 22796
  Collected rows 16384-18432 / 22796
  Collected rows 18432-20480 / 22796
  Collected rows 20480-22528 / 22796
  Collected rows 22528-22796 / 22796
Combining table...
Done!
Classified raster: /home/gisuser/code/data/sites_selection_data/output/forest_typology_class.tif
Table exported: /home/gisuser/code/data/sites_selection_data/output/forest_typology_table.csv
Total classified pixels: 692128
  Mature (1): 33739
  Non-restored (2): 229063
  Natural Regen (4): 429326


In [6]:
import numpy as np
import rasterio
from rasterio.windows import from_bounds

# Check GLAD directly
with rasterio.open(paths['glad_height']) as src:
    print("=== GLAD ===")
    print("CRS:", src.crs)
    print("Shape:", src.height, "x", src.width)
    print("Transform:", src.transform)
    print("Bounds:", src.bounds)
    print("dtype:", src.dtypes[0])
    print("NoData:", src.nodata)
    
    # Read full raster
    data = src.read(1)
    print("Unique values (first 20):", np.unique(data)[:20])
    print("Non-zero pixels:", (data != 0).sum())

# Now check what we get when reading at typology extent
with rasterio.open(paths['typology']) as ref:
    ref_bounds = ref.bounds

with rasterio.open(paths['glad_height']) as src:
    win = from_bounds(*ref_bounds, transform=src.transform)
    win = win.round_offsets().round_lengths()
    print("=== GLAD at typology extent ===")
    print("Window:", win)
    data_at_extent = src.read(1, window=win)
    print("Unique values (first 20):", np.unique(data_at_extent)[:20])
    print("Non-zero pixels:", (data_at_extent != 0).sum())


=== GLAD ===
CRS: ESRI:102033
Shape: 22796 x 29055
Transform: | 31.89, 0.00, 956371.64|
| 0.00,-31.89, 1288790.51|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=956371.638759929, bottom=561893.2195062793, right=1882850.0036588889, top=1288790.5110649331)
dtype: uint8
NoData: 15.0
Unique values (first 20): [ 0  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
Non-zero pixels: 96984727
=== GLAD at typology extent ===
Window: Window(col_off=0, row_off=0, width=29055, height=22796)
Unique values (first 20): [ 0  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]
Non-zero pixels: 96984727


In [ ]:
with rasterio.open(glad_path) as src:
    data = src.read(1)
    print("Max value:", data.max())
    print("Non-zero:", (data != 0).sum())


: 